## 1. Import Library and Setup Configuration

In [ ]:
# import necessary libraries
import os
import sys
import json
import re
import subprocess
import random
import warnings

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModel,
    Blip2QFormerModel,
    Blip2QFormerConfig,
)
from transformers.models.blip_2.modeling_blip_2 import Blip2TextEmbeddings
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

from sklearn.metrics import (
    classification_report,
    f1_score,
    balanced_accuracy_score,
    average_precision_score,
    precision_recall_fscore_support,
)

from health_multimodal.image import get_image_inference
from health_multimodal.image.utils import ImageModelType

warnings.filterwarnings("ignore")

In [ ]:
# setup reproducibility
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

In [ ]:
# setup global configuration
config = {
    "experiment": "exp_002",                     # "exp_001" or "exp_002"
    "stage": "stage_meeting",

    "data": {
        "train_path": r"D:\VLM-Research_Task-C\data\processed\chexpert_plus\train.parquet",
        "dev_path": r"D:\VLM-Research_Task-C\data\processed\chexpert_plus\dev_internal.parquet",
        "test_path": r"D:\VLM-Research_Task-C\data\processed\chexpert_plus\test.parquet",
        "image_col": "actual_image_path",
        "label_suffix": "_label",
        "target_col": "target_report_text",
        "max_text_length": 512
    },

    "stage_a": {
        "type": "qformer",                       # "lightweight" or "qformer"
        "checkpoint_dir": r"D:\VLM-Research_Task-C\output\exp_002\stage_a\checkpoints",
        "checkpoint_file": "stage_a_best.pt",
        "llm_hidden_dim": 1024,
        "num_queries": 32,
        "qformer_hidden_size": 768,
        "vision_dim": 512,                       # will be overwritten at runtime
    },

    "stage_b": {
        "checkpoint_dir": r"D:\VLM-Research_Task-C\output\exp_001\stage_b\checkpoints",
        "checkpoint_file": "stage_b_best_f1.pt",
        "tuning_dir": r"D:\VLM-Research_Task-C\output\exp_001\stage_b\tuning",
        "threshold_file": "best_threshold_per_class_best_f1.json"
    },

    "model": {
        "biogpt_name": "microsoft/biogpt",
        "llm_hidden_dim": 1024,
        "num_queries": 32,
        "dropout": 0.1,
        "lora_r": 8,
        "lora_alpha": 16,
        "lora_dropout": 0.05,
        "lora_target_modules": ["q_proj", "v_proj"]
    },

    "train": {
        "batch_size": 8,
        "num_epochs": 20,
        "lr": 5.0e-5,
        "weight_decay": 1.0e-4,
        "grad_clip_norm": 1.0,
        "early_stopping_patience": 3,
        "num_workers": 0,
        "seed": 42,
        "max_length": 512,
        "warmup_steps": 100,
        "checkpoint_dir": r"D:\VLM-Research_Task-C\output\exp_002\stage_meeting\checkpoints",
        "log_dir": r"D:\VLM-Research_Task-C\output\exp_002\stage_meeting\logs",
        "plot_dir": r"D:\VLM-Research_Task-C\output\exp_002\stage_meeting\plots"
    },

    "inference": {
        "num_beams": 4,
        "max_new_tokens": 512,
        "do_sample": False
    },

    "evaluation": {
        "bootstrap_n": 1000,
        "bootstrap_ci": 0.95
    },

    "chexbert": {
        "repo_dir": r"D:\VLM-Research_Task-C\external\chexbert",
        "checkpoint": r"D:\VLM-Research_Task-C\external\chexbert\pretrained_weights\chexbert.pth"
    },

    "device": "cuda"
}

# create output directories
for key in ["checkpoint_dir", "log_dir", "plot_dir"]:
    os.makedirs(config["train"][key], exist_ok=True)

In [ ]:
# define device, if GPU doesn't exist, it'll fallback to CPU usage
device = torch.device(config['device'] if torch.cuda.is_available() else 'cpu')
print(f"using device: {device}")

## 2. Load Dataset

In [ ]:
df_train = pd.read_parquet(config['data']['train_path'])
df_dev = pd.read_parquet(config['data']['dev_path'])
df_test = pd.read_parquet(config['data']['test_path'])

print(f"train set: {df_train.shape[0]} rows & {df_train.shape[1]} columns")
print(f"dev set: {df_dev.shape[0]} rows & {df_dev.shape[1]} columns")
print(f"test set: {df_test.shape[0]} rows & {df_test.shape[1]} columns")

label_cols = [col for col in df_train.columns if col.endswith(config['data']['label_suffix'])]
print(f"label columns ({len(label_cols)}): {label_cols}")

## 3. Training Utilities

### 3.1 Dataset Class

In [ ]:
class StageMeetingDataset(Dataset):
    def __init__(self, df, image_col, transform=None):
        self.df = df.reset_index(drop=True)
        self.image_col = image_col
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row[self.image_col]
        try:
            image = Image.open(img_path).convert('L')
        except FileNotFoundError:
            print(f"warning: image not found {img_path}, returning zeros.")
            image = Image.new('L', (224, 224))
        if self.transform:
            image = self.transform(image)
        target_text = row["target_report_text"]
        return image, target_text

### 3.2 Load BioViL-T

In [ ]:
# load biovil-t
image_engine = get_image_inference(ImageModelType.BIOVIL_T)
biovilt_transform = image_engine.transform
bio_model = image_engine.model
bio_model.to(device)
bio_model.eval()

for param in bio_model.parameters():
    param.requires_grad = False

print("biovil-t loaded successfully (frozen)")

# detect vision dimension
sample_row = df_train.iloc[0]
sample_img_path = sample_row[config['data']['image_col']]
try:
    sample_img = Image.open(sample_img_path).convert('L')
except FileNotFoundError:
    sample_img = Image.new('L', (224, 224))

sample_pixel_values = biovilt_transform(sample_img).unsqueeze(0).to(device)

with torch.no_grad():
    dummy_out = bio_model(sample_pixel_values)
    if hasattr(dummy_out, 'patch_embeddings'):
        features = dummy_out.patch_embeddings
        print("using patch_embeddings")
    elif hasattr(dummy_out, 'last_hidden_state'):
        features = dummy_out.last_hidden_state
        print("using last_hidden_state")
    else:
        raise RuntimeError("cannot extract features from biovil-t output")

    if features.dim() == 3:
        vision_dim = features.shape[-1]
    elif features.dim() == 4:
        vision_dim = features.shape[1]
    else:
        vision_dim = features.shape[-1]

    if vision_dim < 100:
        print(f"warning: detected vision_dim={vision_dim}, forcing to 512")
        vision_dim = 512

config['vision_dim'] = vision_dim
config['stage_a']['vision_dim'] = vision_dim
print(f"vision hidden dimension: {vision_dim}")

# feature extractor
def extract_spatial_features(backbone, images):
    with torch.no_grad():
        out = backbone(images)
        if hasattr(out, 'patch_embeddings'):
            features = out.patch_embeddings
            if features.dim() == 4:
                B, C, H, W = features.shape
                features = features.flatten(2).transpose(1, 2)
            return features
        elif hasattr(out, 'last_hidden_state'):
            features = out.last_hidden_state
            if features.dim() == 4:
                B, C, H, W = features.shape
                features = features.flatten(2).transpose(1, 2)
            return features
        else:
            raise ValueError("biovil-t output format not recognized")

### 3.3 Define and Load Classifier Module

In [ ]:
# define classifier model
class StageBClassifier(nn.Module):
    def __init__(self, config, backbone, vision_dim, freeze_backbone=False):
        super().__init__()
        self.backbone = backbone
        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False

        self.hidden_dim = config['model']['hidden_dim']
        self.num_classes = config['model']['num_pathologies']
        self.dropout_rate = config['model']['dropout']
        self.pooling = config['model'].get('pooling', 'attention')

        in_features = vision_dim

        if self.pooling == "attention":
            self.attn_pool = nn.Linear(in_features, 1)

        if self.hidden_dim is not None:
            self.classifier = nn.Sequential(
                nn.Dropout(self.dropout_rate),
                nn.Linear(in_features, self.hidden_dim),
                nn.ReLU(),
                nn.Dropout(self.dropout_rate),
                nn.Linear(self.hidden_dim, self.num_classes)
            )
        else:
            self.classifier = nn.Sequential(
                nn.Dropout(self.dropout_rate),
                nn.Linear(in_features, self.num_classes)
            )

    def forward(self, x):
        out = self.backbone(x)

        if hasattr(out, 'patch_embeddings'):
            features = out.patch_embeddings
        elif hasattr(out, 'last_hidden_state'):
            features = out.last_hidden_state
        else:
            features = out

        if not isinstance(features, torch.Tensor):
            raise TypeError(f"expected tensor, got {type(features)}")

        if features.dim() == 3:
            if self.pooling == "attention":
                attn_scores = self.attn_pool(features)
                attn_weights = torch.softmax(attn_scores, dim=1)
                pooled = (features * attn_weights).sum(dim=1)
            else:
                pooled = features.mean(dim=1)
        elif features.dim() == 4:
            if self.pooling == "attention":
                b, c, h, w = features.shape
                flat = features.flatten(2).transpose(1, 2)
                attn_scores = self.attn_pool(flat)
                attn_weights = torch.softmax(attn_scores, dim=1)
                pooled = (flat * attn_weights).sum(dim=1)
            else:
                pooled = features.mean(dim=[2, 3])
        else:
            pooled = features.flatten(1) if features.dim() > 2 else features

        logits = self.classifier(pooled)
        return logits

In [ ]:
# load stage-b classifier and thresholds
def load_stage_b(config, device):
    image_engine_b = get_image_inference(ImageModelType.BIOVIL_T)
    bio_model_b = image_engine_b.model
    bio_model_b.to(device)
    bio_model_b.eval()

    model_cfg = {
        'model': {
            'hidden_dim': None,
            'num_pathologies': 14,
            'dropout': 0.1,
            'pooling': 'attention'
        }
    }
    model = StageBClassifier(
        config=model_cfg,
        backbone=bio_model_b,
        vision_dim=config['vision_dim'],
        freeze_backbone=True
    )
    checkpoint_path = os.path.join(config['stage_b']['checkpoint_dir'], config['stage_b']['checkpoint_file'])
    state = torch.load(checkpoint_path, map_location=device)
    if 'model_state_dict' in state:
        model.load_state_dict(state['model_state_dict'])
    else:
        model.load_state_dict(state)
    model.to(device)
    model.eval()
    for param in model.parameters():
        param.requires_grad = False

    threshold_path = os.path.join(config['stage_b']['tuning_dir'], config['stage_b']['threshold_file'])
    with open(threshold_path, 'r') as f:
        per_class_th = json.load(f)

    print(f"stage-b classifier loaded from {checkpoint_path}")
    print("per-class thresholds loaded")
    return model, per_class_th

stage_b_model, per_class_th = load_stage_b(config, device)

### 3.4 Define and Load Alignment Module

In [ ]:
# define lightweight AlignmentModule (same as in stage_a exp_001)
class AlignmentModuleLightweight(nn.Module):
    def __init__(self, visual_feature_dim, hidden_dim, num_queries=32, num_layers=2, num_heads=8, dropout=0.1):
        super().__init__()
        self.num_queries = num_queries
        self.hidden_dim = hidden_dim
        self.query_embeddings = nn.Parameter(torch.randn(num_queries, hidden_dim) * 0.02)
        self.visual_proj = nn.Linear(visual_feature_dim, hidden_dim)
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim * 4,
            dropout=dropout,
            batch_first=True
        )
        self.cross_attention = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        self.output_norm = nn.LayerNorm(hidden_dim)

    def forward(self, visual_features):
        batch_size = visual_features.size(0)
        memory = self.visual_proj(visual_features)
        queries = self.query_embeddings.unsqueeze(0).expand(batch_size, -1, -1)
        soft_tokens = self.cross_attention(tgt=queries, memory=memory)
        return self.output_norm(soft_tokens)

# define QFormer wrapper (same as in stage_a exp_002)
class QFormerWrapper(nn.Module):
    def __init__(self, qformer, query_tokens, text_embeddings, llm_proj):
        super().__init__()
        self.qformer = qformer
        self.query_tokens = query_tokens
        self.text_embeddings = text_embeddings
        self.llm_proj = llm_proj

    def forward(self, visual_features):
        batch_size = visual_features.size(0)
        queries = self.query_tokens.expand(batch_size, -1, -1)
        image_atts = torch.ones(visual_features.size()[:-1], dtype=torch.long, device=visual_features.device)
        output = self.qformer(
            query_embeds=queries,
            encoder_hidden_states=visual_features,
            encoder_attention_mask=image_atts,
            return_dict=True,
        )
        soft_tokens = output.last_hidden_state  # (B, 32, 768)
        soft_tokens = self.llm_proj(soft_tokens)  # (B, 32, 1024)
        return soft_tokens

In [ ]:
# load stage-a alignment module
def load_stage_a(config, device):
    stage_a_type = config['stage_a']['type']
    checkpoint_path = os.path.join(config['stage_a']['checkpoint_dir'], config['stage_a']['checkpoint_file'])
    checkpoint = torch.load(checkpoint_path, map_location=device)

    if stage_a_type == "lightweight":
        alignment_module = AlignmentModuleLightweight(
            visual_feature_dim=config['stage_a']['vision_dim'],
            hidden_dim=config['stage_a']['llm_hidden_dim'],
            num_queries=config['stage_a']['num_queries'],
            num_layers=2,
            num_heads=8,
            dropout=0.1
        )
        alignment_module.load_state_dict(checkpoint['alignment_module_state_dict'])
        llm_proj = nn.Identity()
        print("stage-a: lightweight alignment module loaded")

    elif stage_a_type == "qformer":
        qformer_config = Blip2QFormerConfig(
            vocab_size=30522,
            hidden_size=768,
            num_hidden_layers=12,
            num_attention_heads=12,
            intermediate_size=3072,
            cross_attention_frequency=2,
            encoder_hidden_size=config['stage_a']['vision_dim'],
            use_qformer_text_input=True,
        )
        qformer = Blip2QFormerModel(qformer_config).to(device)
        qformer.load_state_dict(checkpoint['qformer_state_dict'])

        query_tokens = checkpoint['query_tokens'].to(device)

        qformer_text_embeddings = Blip2TextEmbeddings(qformer_config).to(device)
        if 'qformer_text_embeddings_state_dict' in checkpoint:
            qformer_text_embeddings.load_state_dict(checkpoint['qformer_text_embeddings_state_dict'])

        llm_proj = nn.Linear(768, config['stage_a']['llm_hidden_dim']).to(device)
        if 'llm_proj_state_dict' in checkpoint:
            llm_proj.load_state_dict(checkpoint['llm_proj_state_dict'])

        alignment_module = QFormerWrapper(qformer, query_tokens, qformer_text_embeddings, llm_proj)
        print("stage-a: q-former alignment module loaded")

    else:
        raise ValueError(f"unknown stage_a type: {stage_a_type}")

    alignment_module.to(device)
    alignment_module.eval()
    for param in alignment_module.parameters():
        param.requires_grad = False

    return alignment_module

alignment_module = load_stage_a(config, device)

### 3.5 Helpers

In [ ]:
# getter for chexpert labels from stage-b
def get_chexpert_labels(images, stage_b_model, per_class_th, label_cols, device):
    no_finding_idx = None
    for i, col in enumerate(label_cols):
        if col.lower().startswith('no finding'):
            no_finding_idx = i
            break
    if no_finding_idx is None:
        no_finding_idx = 0
    other_indices = [i for i in range(len(label_cols)) if i != no_finding_idx]

    with torch.no_grad():
        logits = stage_b_model(images)
        probs = torch.sigmoid(logits).cpu().numpy()

    prompts = []
    for row_probs in probs:
        active = []
        for i in other_indices:
            col_name = label_cols[i]
            th = per_class_th.get(col_name, 0.5)
            if row_probs[i] >= th:
                active.append(col_name.replace('_label', ''))
        if len(active) == 0:
            prompts.append("chexpert findings: no significant findings")
        else:
            prompts.append(f"chexpert findings: {', '.join(active)}")
    return prompts

In [ ]:
# define precomputer for stage meeting
def precompute_stage_data(dataloader, alignment_module, stage_b_model, per_class_th, label_cols, device, desc="precomputing"):
    soft_tokens_list = []
    chexpert_strings = []
    target_texts = []

    for images, texts in tqdm(dataloader, desc=desc):
        images = images.to(device)
        with torch.no_grad():
            visual_features = extract_spatial_features(bio_model, images)
            soft_tokens = alignment_module(visual_features)
        soft_tokens_list.append(soft_tokens.cpu().numpy())

        chexpert = get_chexpert_labels(images, stage_b_model, per_class_th, label_cols, device)
        chexpert_strings.extend(chexpert)
        target_texts.extend(texts)

    soft_tokens_list = np.vstack(soft_tokens_list)
    return soft_tokens_list, chexpert_strings, target_texts

# create dataloaders
train_ds = StageMeetingDataset(df_train, config['data']['image_col'], transform=biovilt_transform)
dev_ds = StageMeetingDataset(df_dev, config['data']['image_col'], transform=biovilt_transform)
test_ds = StageMeetingDataset(df_test, config['data']['image_col'], transform=biovilt_transform)
train_loader = DataLoader(train_ds, batch_size=config['train']['batch_size'], shuffle=True, num_workers=config['train']['num_workers'], pin_memory=True)
dev_loader = DataLoader(dev_ds, batch_size=config['train']['batch_size'], shuffle=False, num_workers=config['train']['num_workers'], pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=config['train']['batch_size'], shuffle=False, num_workers=config['train']['num_workers'], pin_memory=True)
print(f"train batches: {len(train_loader)}, dev batches: {len(dev_loader)}, test batches: {len(test_loader)}")

# precompute
print("precomputing stage-a and stage-b outputs for train set...")
train_soft, train_chexpert, train_targets = precompute_stage_data(
    train_loader, alignment_module, stage_b_model, per_class_th, label_cols, device, "train precompute"
)
print("precomputing stage-a and stage-b outputs for dev set...")
dev_soft, dev_chexpert, dev_targets = precompute_stage_data(
    dev_loader, alignment_module, stage_b_model, per_class_th, label_cols, device, "dev precompute"
)
print("precomputing stage-a and stage-b outputs for test set...")
test_soft, test_chexpert, test_targets = precompute_stage_data(
    test_loader, alignment_module, stage_b_model, per_class_th, label_cols, device, "test precompute"
)

### 3.6 Pre-computed Dataset for Training Needs

In [ ]:
class StageMeetingPrecomputedDataset(Dataset):
    def __init__(self, soft_tokens, chexpert_strings, target_texts):
        self.soft_tokens = soft_tokens
        self.chexpert_strings = chexpert_strings
        self.target_texts = target_texts

    def __len__(self):
        return len(self.target_texts)

    def __getitem__(self, idx):
        return {
            'soft_tokens': torch.tensor(self.soft_tokens[idx], dtype=torch.float32),
            'chexpert': self.chexpert_strings[idx],
            'target': self.target_texts[idx]
        }

train_ds_pre = StageMeetingPrecomputedDataset(train_soft, train_chexpert, train_targets)
dev_ds_pre = StageMeetingPrecomputedDataset(dev_soft, dev_chexpert, dev_targets)
test_ds_pre = StageMeetingPrecomputedDataset(test_soft, test_chexpert, test_targets)
train_loader_pre = DataLoader(train_ds_pre, batch_size=config['train']['batch_size'], shuffle=True, num_workers=config['train']['num_workers'], pin_memory=True)
dev_loader_pre = DataLoader(dev_ds_pre, batch_size=config['train']['batch_size'], shuffle=False, num_workers=config['train']['num_workers'], pin_memory=True)
test_loader_pre = DataLoader(test_ds_pre, batch_size=config['train']['batch_size'], shuffle=False, num_workers=config['train']['num_workers'], pin_memory=True)

### 3.7 Load BioGPT with LoRA

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(config['model']['biogpt_name'])
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    config['model']['biogpt_name'],
    torch_dtype=torch.float16,
    device_map="auto" if device.type == 'cuda' else None,
)
base_model.config.pad_token_id = tokenizer.pad_token_id

lora_config = LoraConfig(
    r=config['model']['lora_r'],
    lora_alpha=config['model']['lora_alpha'],
    target_modules=config['model']['lora_target_modules'],
    lora_dropout=config['model']['lora_dropout'],
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

### 3.8 Utilities

In [ ]:
def build_prompt(chexpert_string, task_instruction="generate radiology report."):
    return f"{chexpert_string} {task_instruction}"


def collate_fn(batch):
    soft_tokens = torch.stack([item['soft_tokens'] for item in batch])
    chexpert = [item['chexpert'] for item in batch]
    targets = [item['target'] for item in batch]
    return soft_tokens, chexpert, targets


def build_labels_with_prompt_mask(tokenizer, prompts, input_ids, attention_mask, device):
    """
    [FIX] mask token prompt (chexpert string + instruction) dan padding jadi -100 (ignore_index),
    supaya loss HANYA dihitung atas token laporan target - bukan atas prompt yang sudah given.
    """
    prompt_lengths = [
        len(tokenizer(p, add_special_tokens=True).input_ids) for p in prompts
    ]
    labels = input_ids.clone()
    for i, plen in enumerate(prompt_lengths):
        labels[i, :plen] = -100
    labels[attention_mask == 0] = -100  # padding token juga di-ignore
    return labels.to(device)


def train_epoch(model, dataloader, optimizer, scheduler, device, max_length=512):
    model.train()
    total_loss = 0
    for soft_tokens, chexpert_strings, targets in tqdm(dataloader, desc="training"):
        soft_tokens = soft_tokens.to(device)

        prompts = [build_prompt(s) for s in chexpert_strings]
        full_texts = [p + " " + t for p, t in zip(prompts, targets)]

        tokenized = tokenizer(full_texts, padding=True, truncation=True, max_length=max_length, return_tensors="pt")
        input_ids = tokenized.input_ids.to(device)
        attention_mask = tokenized.attention_mask.to(device)

        # [FIX] pakai API resmi HF (get_input_embeddings), bukan path atribut hardcoded yang rapuh
        with torch.no_grad():
            text_embeds = model.get_input_embeddings()(input_ids)

        # [FIX] samakan dtype soft_tokens (fp32) dengan text_embeds (fp16, karena base_model di-load fp16)
        soft_tokens = soft_tokens.to(text_embeds.dtype)
        inputs_embeds = torch.cat([soft_tokens, text_embeds], dim=1)

        soft_mask = torch.ones(soft_tokens.size(0), soft_tokens.size(1), dtype=torch.long, device=device)
        full_attention_mask = torch.cat([soft_mask, attention_mask], dim=1)

        # [FIX] label prompt di-mask -100, dan di-pad -100 utk posisi soft_tokens (panjang harus match inputs_embeds)
        text_labels = build_labels_with_prompt_mask(tokenizer, prompts, input_ids, attention_mask, device)
        soft_labels = torch.full((soft_tokens.size(0), soft_tokens.size(1)), -100, dtype=torch.long, device=device)
        labels = torch.cat([soft_labels, text_labels], dim=1)

        outputs = model(
            inputs_embeds=inputs_embeds,
            attention_mask=full_attention_mask,
            labels=labels,
        )
        loss = outputs.loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), config['train']['grad_clip_norm'])
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        total_loss += loss.item()

    return total_loss / len(dataloader)


def evaluate(model, dataloader, device, max_length=512):
    """
    [FIX - KRITIS] sebelumnya inputs_embeds utk model.generate() ikut menyertakan teks target
    (laporan ground truth), sehingga model "melihat" jawabannya sebelum menulis - semua metrik
    evaluasi (BLEU/ROUGE/METEOR/BERTScore/CheXbert CE) jadi tidak valid.
    sekarang dipisah tegas:
      - loss (teacher-forced): pakai prompt+target, label prompt di-mask -100 (sama seperti train_epoch)
      - generate (zero-shot): pakai prompt SAJA, tanpa target sama sekali
    """
    model.eval()
    total_loss = 0
    all_generated = []
    all_targets = []
    with torch.no_grad():
        for soft_tokens, chexpert_strings, targets in tqdm(dataloader, desc="evaluating"):
            soft_tokens = soft_tokens.to(device)
            prompts = [build_prompt(s) for s in chexpert_strings]

            # ---------- loss (teacher-forced, prompt + target) ----------
            full_texts = [p + " " + t for p, t in zip(prompts, targets)]
            tokenized_full = tokenizer(full_texts, padding=True, truncation=True, max_length=max_length, return_tensors="pt")
            input_ids_full = tokenized_full.input_ids.to(device)
            attention_mask_full = tokenized_full.attention_mask.to(device)

            text_embeds_full = model.get_input_embeddings()(input_ids_full)
            soft_tokens_fp = soft_tokens.to(text_embeds_full.dtype)
            inputs_embeds_full = torch.cat([soft_tokens_fp, text_embeds_full], dim=1)

            soft_mask = torch.ones(soft_tokens.size(0), soft_tokens.size(1), dtype=torch.long, device=device)
            full_attention_mask = torch.cat([soft_mask, attention_mask_full], dim=1)

            text_labels = build_labels_with_prompt_mask(tokenizer, prompts, input_ids_full, attention_mask_full, device)
            soft_labels = torch.full((soft_tokens.size(0), soft_tokens.size(1)), -100, dtype=torch.long, device=device)
            labels = torch.cat([soft_labels, text_labels], dim=1)

            outputs = model(
                inputs_embeds=inputs_embeds_full,
                attention_mask=full_attention_mask,
                labels=labels,
            )
            total_loss += outputs.loss.item()

            # ---------- generate (zero-shot, prompt SAJA - tanpa target) ----------
            tokenized_prompt = tokenizer(prompts, padding=True, truncation=True, max_length=max_length, return_tensors="pt")
            input_ids_prompt = tokenized_prompt.input_ids.to(device)
            attention_mask_prompt = tokenized_prompt.attention_mask.to(device)

            text_embeds_prompt = model.get_input_embeddings()(input_ids_prompt)
            soft_tokens_gen = soft_tokens.to(text_embeds_prompt.dtype)
            inputs_embeds_prompt = torch.cat([soft_tokens_gen, text_embeds_prompt], dim=1)
            gen_attention_mask = torch.cat([soft_mask, attention_mask_prompt], dim=1)

            gen_outputs = model.generate(
                inputs_embeds=inputs_embeds_prompt,
                attention_mask=gen_attention_mask,
                max_new_tokens=config['inference']['max_new_tokens'],
                num_beams=config['inference']['num_beams'],
                do_sample=config['inference']['do_sample'],
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
            generated_texts = tokenizer.batch_decode(gen_outputs, skip_special_tokens=True)
            all_generated.extend(generated_texts)
            all_targets.extend(targets)

    return total_loss / len(dataloader), all_generated, all_targets


## 4. Train Loop

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=config['train']['lr'], weight_decay=config['train']['weight_decay'])
total_steps = len(train_loader_pre) * config['train']['num_epochs']
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps)

best_val_loss = float('inf')
early_stop_counter = 0
train_losses, val_losses = [], []

print("starting training...")
for epoch in range(1, config['train']['num_epochs'] + 1):
    train_loss = train_epoch(model, train_loader_pre, optimizer, scheduler, device, max_length=config['train']['max_length'])
    train_losses.append(train_loss)

    val_loss, _, _ = evaluate(model, dev_loader_pre, device, max_length=config['train']['max_length'])
    val_losses.append(val_loss)

    print(f"epoch {epoch}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        early_stop_counter = 0
        checkpoint_path = os.path.join(config['train']['checkpoint_dir'], "stage_meeting_best.pt")
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_val_loss': best_val_loss,
        }, checkpoint_path)
        print(f"  -> new best model saved (val_loss={val_loss:.4f})")
    else:
        early_stop_counter += 1
        if early_stop_counter >= config['train']['early_stopping_patience']:
            print(f"early stopping triggered at epoch {epoch}")
            break

print(f"\ntraining finished. best validation loss: {best_val_loss:.4f}")

## 5. Evaluation

In [ ]:
# loss history curve
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='train loss')
plt.plot(val_losses, label='val loss')
plt.xlabel('epoch')
plt.ylabel('loss')
plt.legend()
plt.title('training & validation loss (cross-entropy)')
plt.grid(True)
plt.savefig(os.path.join(config['train']['plot_dir'], 'loss_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# define functions for evaluation

# load best model
best_model_path = os.path.join(config['train']['checkpoint_dir'], "stage_meeting_best.pt")
checkpoint = torch.load(best_model_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print("evaluating on test set...")
test_loss, generated, targets = evaluate(model, test_loader_pre, device,  max_length=config['train']['max_length'])
print(f"test loss: {test_loss:.4f}")

In [ ]:
# split generated and target texts into findings / impression / full
def extract_findings_impression(text):
    findings_match = re.search(r'findings:\s*(.*?)(?=\s+impression:|$)', text, re.IGNORECASE | re.DOTALL)
    impression_match = re.search(r'impression:\s*(.*)$', text, re.IGNORECASE | re.DOTALL)
    findings = findings_match.group(1).strip() if findings_match else ""
    impression = impression_match.group(1).strip() if impression_match else ""
    return findings, impression

gen_findings, gen_impression = [], []
ref_findings, ref_impression = [], []
for gen, ref in zip(generated, targets):
    gf, gi = extract_findings_impression(gen)
    rf, ri = extract_findings_impression(ref)
    gen_findings.append(gf)
    gen_impression.append(gi)
    ref_findings.append(rf)
    ref_impression.append(ri)

gen_full = generated
ref_full = targets

In [ ]:
try:
    from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
    from rouge_score import rouge_scorer
    from nltk.translate.meteor_score import meteor_score
    from bert_score import score as bert_score
except ImportError:
    print("install nltk, rouge-score, bert-score for nlg metrics")
    nlg_metrics = {}
else:
    def compute_nlg_scores(preds, refs):
        smoothie = SmoothingFunction().method4
        bleu1 = np.mean([corpus_bleu([[r.split()]], [p.split()], weights=(1,0,0,0), smoothing_function=smoothie) for p, r in zip(preds, refs)])
        bleu2 = np.mean([corpus_bleu([[r.split()]], [p.split()], weights=(0.5,0.5,0,0), smoothing_function=smoothie) for p, r in zip(preds, refs)])
        bleu3 = np.mean([corpus_bleu([[r.split()]], [p.split()], weights=(0.33,0.33,0.33,0), smoothing_function=smoothie) for p, r in zip(preds, refs)])
        bleu4 = np.mean([corpus_bleu([[r.split()]], [p.split()], weights=(0.25,0.25,0.25,0.25), smoothing_function=smoothie) for p, r in zip(preds, refs)])
        scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
        rouge_scores = [scorer.score(r, p) for p, r in zip(preds, refs)]
        rouge1 = np.mean([s['rouge1'].fmeasure for s in rouge_scores])
        rouge2 = np.mean([s['rouge2'].fmeasure for s in rouge_scores])
        rougeL = np.mean([s['rougeL'].fmeasure for s in rouge_scores])
        meteor_scores = [meteor_score([r], p) for p, r in zip(preds, refs)]
        meteor = np.mean(meteor_scores)
        P, R, F1 = bert_score(preds, refs, lang='en', verbose=False)
        bertscore = F1.mean().item()
        return {'bleu1': bleu1, 'bleu2': bleu2, 'bleu3': bleu3, 'bleu4': bleu4,
                'rouge1': rouge1, 'rouge2': rouge2, 'rougeL': rougeL,
                'meteor': meteor, 'bertscore': bertscore}

    nlg_metrics = {
        'findings': compute_nlg_scores(gen_findings, ref_findings),
        'impression': compute_nlg_scores(gen_impression, ref_impression),
        'full': compute_nlg_scores(gen_full, ref_full)
    }

print("\n=== nlg metrics by section ===")
for section, metrics in nlg_metrics.items():
    print(f"\n{section.upper()}:")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")

In [ ]:
CHEXBERT_DIR = config['chexbert']['repo_dir']
CHEXBERT_CKPT = config['chexbert']['checkpoint']

if not os.path.exists(CHEXBERT_DIR):
    print("cloning chexbert repository...")
    subprocess.run(['git', 'clone', 'https://github.com/stanfordmlgroup/CheXbert.git', CHEXBERT_DIR], check=True)
    print("chexbert cloned.")
else:
    print("chexbert directory exists.")

def load_chexbert(model_path, device):
    import sys
    sys.path.insert(0, CHEXBERT_DIR)
    from chexbert.chexbert import CheXbert
    model = CheXbert()
    state_dict = torch.load(model_path, map_location=device)
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()
    print(f"chexbert model loaded from {model_path}")
    return model

chexbert_model = load_chexbert(CHEXBERT_CKPT, device)

def binarize_chexbert_labels(df, label_cols, uncertain_as_positive=True):
    binary = []
    for col in label_cols:
        data = df[col].copy()
        data = data.fillna(0)
        if uncertain_as_positive:
            data = data.replace(-1.0, 1.0)
        else:
            data = data.replace(-1.0, 0.0)
        data = data.clip(0, 1)
        binary.append(data.values.astype(np.float32))
    return np.column_stack(binary)

def predict_chexbert_batch(texts, model, batch_size=32):
    all_probs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        with torch.no_grad():
            if hasattr(model, 'predict'):
                logits = model.predict(batch)
            else:
                raise NotImplementedError("chexbert inference not implemented")
            probs = torch.sigmoid(torch.tensor(logits)).numpy()
        all_probs.append(probs)
    return np.vstack(all_probs)

def compute_ce_metrics(pred_probs, ref_binary, threshold=0.5):
    pred_bin = (pred_probs > threshold).astype(int)
    ref_bin = ref_binary.astype(int)

    p_macro, r_macro, f_macro, _ = precision_recall_fscore_support(
        ref_bin, pred_bin, average='macro', zero_division=0
    )
    p_micro, r_micro, f_micro, _ = precision_recall_fscore_support(
        ref_bin, pred_bin, average='micro', zero_division=0
    )
    p_weighted, r_weighted, f_weighted, _ = precision_recall_fscore_support(
        ref_bin, pred_bin, average='weighted', zero_division=0
    )

    n_classes = ref_bin.shape[1]
    pr_auc_macro = np.mean([
        average_precision_score(ref_bin[:, i], pred_probs[:, i])
        for i in range(n_classes)
    ])
    pr_auc_micro = average_precision_score(ref_bin.ravel(), pred_probs.ravel())

    bal_acc = balanced_accuracy_score(ref_bin, pred_bin)

    return {
        'precision_macro': p_macro,
        'recall_macro': r_macro,
        'f1_macro': f_macro,
        'precision_micro': p_micro,
        'recall_micro': r_micro,
        'f1_micro': f_micro,
        'precision_weighted': p_weighted,
        'recall_weighted': r_weighted,
        'f1_weighted': f_weighted,
        'pr_auc_macro': pr_auc_macro,
        'pr_auc_micro': pr_auc_micro,
        'balanced_accuracy': bal_acc,
    }

# prepare ground truth
label_cols_full = [col for col in df_test.columns if col.endswith('_label')]
label_cols_x = [col.replace('_label', '_x') for col in label_cols_full]
label_cols_y = [col.replace('_label', '_y') for col in label_cols_full]

ref_binary_full = df_test[label_cols_full].values.astype(np.float32)
ref_binary_findings = binarize_chexbert_labels(df_test, label_cols_x, uncertain_as_positive=True)
ref_binary_impression = binarize_chexbert_labels(df_test, label_cols_y, uncertain_as_positive=True)

# run chexbert inference
print("\nrunning chexbert inference on generated texts...")
gen_probs_findings = predict_chexbert_batch(gen_findings, chexbert_model)
gen_probs_impression = predict_chexbert_batch(gen_impression, chexbert_model)
gen_probs_full = predict_chexbert_batch(gen_full, chexbert_model)

print("running chexbert inference on reference texts...")
ref_probs_findings = predict_chexbert_batch(ref_findings, chexbert_model)
ref_probs_impression = predict_chexbert_batch(ref_impression, chexbert_model)
ref_probs_full = predict_chexbert_batch(ref_full, chexbert_model)

# compute metrics
ce_results = {
    'findings': compute_ce_metrics(gen_probs_findings, ref_binary_findings),
    'impression': compute_ce_metrics(gen_probs_impression, ref_binary_impression),
    'full': compute_ce_metrics(gen_probs_full, ref_binary_full),
}

print("\n" + "="*60)
print("clinical efficacy (chexbert) by section")
print("="*60)
for section, metrics in ce_results.items():
    print(f"\n{section.upper()} SECTION:")
    print(f"  precision (macro / micro / weighted): {metrics['precision_macro']:.4f} / {metrics['precision_micro']:.4f} / {metrics['precision_weighted']:.4f}")
    print(f"  recall    (macro / micro / weighted): {metrics['recall_macro']:.4f} / {metrics['recall_micro']:.4f} / {metrics['recall_weighted']:.4f}")
    print(f"  f1        (macro / micro / weighted): {metrics['f1_macro']:.4f} / {metrics['f1_micro']:.4f} / {metrics['f1_weighted']:.4f}")
    print(f"  pr-auc (macro / micro): {metrics['pr_auc_macro']:.4f} / {metrics['pr_auc_micro']:.4f}")
    print(f"  balanced accuracy: {metrics['balanced_accuracy']:.4f}")


In [ ]:
# clinical efficacy (chexbert-based) per section
CHEXBERT_DIR = config['chexbert']['repo_dir']
CHEXBERT_CKPT = config['chexbert']['checkpoint']

if not os.path.exists(CHEXBERT_DIR):
    print("cloning chexbert repository...")
    subprocess.run(['git', 'clone', 'https://github.com/stanfordmlgroup/CheXbert.git', CHEXBERT_DIR], check=True)
    print("chexbert cloned.")
else:
    print("chexbert directory exists.")

def load_chexbert(model_path, device):
    import sys
    sys.path.insert(0, CHEXBERT_DIR)
    from chexbert.chexbert import CheXbert
    model = CheXbert()
    state_dict = torch.load(model_path, map_location=device)
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()
    print(f"chexbert model loaded from {model_path}")
    return model

chexbert_model = load_chexbert(CHEXBERT_CKPT, device)

def binarize_chexbert_labels(df, label_cols, uncertain_as_positive=True):
    binary = []
    for col in label_cols:
        data = df[col].copy()
        data = data.fillna(0)
        if uncertain_as_positive:
            data = data.replace(-1.0, 1.0)
        else:
            data = data.replace(-1.0, 0.0)
        data = data.clip(0, 1)
        binary.append(data.values.astype(np.float32))
    return np.column_stack(binary)

def predict_chexbert_batch(texts, model, batch_size=32):
    all_probs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        with torch.no_grad():
            if hasattr(model, 'predict'):
                logits = model.predict(batch)
            else:
                raise NotImplementedError("chexbert inference not implemented")
            probs = torch.sigmoid(torch.tensor(logits)).numpy()
        all_probs.append(probs)
    return np.vstack(all_probs)

def compute_ce_metrics(pred_probs, ref_binary, threshold=0.5):
    pred_bin = (pred_probs > threshold).astype(int)
    ref_bin = ref_binary.astype(int)

    p_macro, r_macro, f_macro, _ = precision_recall_fscore_support(
        ref_bin, pred_bin, average='macro', zero_division=0
    )
    p_micro, r_micro, f_micro, _ = precision_recall_fscore_support(
        ref_bin, pred_bin, average='micro', zero_division=0
    )
    p_weighted, r_weighted, f_weighted, _ = precision_recall_fscore_support(
        ref_bin, pred_bin, average='weighted', zero_division=0
    )

    n_classes = ref_bin.shape[1]
    pr_auc_macro = np.mean([
        average_precision_score(ref_bin[:, i], pred_probs[:, i])
        for i in range(n_classes)
    ])
    pr_auc_micro = average_precision_score(ref_bin.ravel(), pred_probs.ravel())

    bal_acc = balanced_accuracy_score(ref_bin, pred_bin)

    return {
        'precision_macro': p_macro,
        'recall_macro': r_macro,
        'f1_macro': f_macro,
        'precision_micro': p_micro,
        'recall_micro': r_micro,
        'f1_micro': f_micro,
        'precision_weighted': p_weighted,
        'recall_weighted': r_weighted,
        'f1_weighted': f_weighted,
        'pr_auc_macro': pr_auc_macro,
        'pr_auc_micro': pr_auc_micro,
        'balanced_accuracy': bal_acc,
    }

# prepare ground truth
label_cols_full = [col for col in df_test.columns if col.endswith('_label')]
label_cols_x = [col.replace('_label', '_x') for col in label_cols_full]
label_cols_y = [col.replace('_label', '_y') for col in label_cols_full]

ref_binary_full = df_test[label_cols_full].values.astype(np.float32)
ref_binary_findings = binarize_chexbert_labels(df_test, label_cols_x, uncertain_as_positive=True)
ref_binary_impression = binarize_chexbert_labels(df_test, label_cols_y, uncertain_as_positive=True)

# run chexbert inference
print("\nrunning chexbert inference on generated texts...")
gen_probs_findings = predict_chexbert_batch(gen_findings, chexbert_model)
gen_probs_impression = predict_chexbert_batch(gen_impression, chexbert_model)
gen_probs_full = predict_chexbert_batch(gen_full, chexbert_model)

print("running chexbert inference on reference texts...")
ref_probs_findings = predict_chexbert_batch(ref_findings, chexbert_model)
ref_probs_impression = predict_chexbert_batch(ref_impression, chexbert_model)
ref_probs_full = predict_chexbert_batch(ref_full, chexbert_model)

# compute metrics
ce_results = {
    'findings': compute_ce_metrics(gen_probs_findings, ref_binary_findings),
    'impression': compute_ce_metrics(gen_probs_impression, ref_binary_impression),
    'full': compute_ce_metrics(gen_probs_full, ref_binary_full),
}

print("\n" + "="*60)
print("clinical efficacy (chexbert) by section")
print("="*60)
for section, metrics in ce_results.items():
    print(f"\n{section.upper()} SECTION:")
    print(f"  precision (macro / micro / weighted): {metrics['precision_macro']:.4f} / {metrics['precision_micro']:.4f} / {metrics['precision_weighted']:.4f}")
    print(f"  recall    (macro / micro / weighted): {metrics['recall_macro']:.4f} / {metrics['recall_micro']:.4f} / {metrics['recall_weighted']:.4f}")
    print(f"  f1        (macro / micro / weighted): {metrics['f1_macro']:.4f} / {metrics['f1_micro']:.4f} / {metrics['f1_weighted']:.4f}")
    print(f"  pr-auc (macro / micro): {metrics['pr_auc_macro']:.4f} / {metrics['pr_auc_micro']:.4f}")
    print(f"  balanced accuracy: {metrics['balanced_accuracy']:.4f}")

In [ ]:
def prepare_qualitative_df(generated_texts, target_texts, max_samples=100):
    data = []
    for gen, ref in zip(generated_texts[:max_samples], target_texts[:max_samples]):
        gen_f, gen_i = extract_findings_impression(gen)
        ref_f, ref_i = extract_findings_impression(ref)
        data.append({
            'generated_full': gen,
            'target_full': ref,
            'generated_findings': gen_f,
            'target_findings': ref_f,
            'generated_impression': gen_i,
            'target_impression': ref_i
        })
    return pd.DataFrame(data)

print("\ngenerating dev set predictions for qualitative analysis...")
dev_loss, dev_generated, dev_targets = evaluate(model, dev_loader_pre, device,  max_length=config['train']['max_length'])
print(f"dev loss: {dev_loss:.4f}")

qualitative_dev_df = prepare_qualitative_df(dev_generated, dev_targets)
qualitative_dev_df.to_csv(os.path.join(config['train']['log_dir'], 'qualitative_dev.csv'), index=False)
print(f"qualitative dev results saved to {config['train']['log_dir']}/qualitative_dev.csv")

qualitative_test_df = prepare_qualitative_df(generated, targets)
qualitative_test_df.to_csv(os.path.join(config['train']['log_dir'], 'qualitative_test.csv'), index=False)
print(f"qualitative test results saved to {config['train']['log_dir']}/qualitative_test.csv")

### 6. Summary of Stage Meeting

In [ ]:
summary = f"""
====================================================================
stage-meeting training summary
====================================================================
experiment: {config['experiment']}
stage: {config['stage']}
device: {device}

dataset:
  - train: {len(df_train)} samples
  - dev: {len(df_dev)} samples
  - test: {len(df_test)} samples

model:
  - biogpt: {config['model']['biogpt_name']}
  - lora_r: {config['model']['lora_r']}
  - lora_alpha: {config['model']['lora_alpha']}
  - lora_target: {config['model']['lora_target_modules']}
  - stage_a_type: {config['stage_a']['type']}
  - stage_b_checkpoint: {config['stage_b']['checkpoint_file']}

training:
  - batch size: {config['train']['batch_size']}
  - learning rate: {config['train']['lr']}
  - weight decay: {config['train']['weight_decay']}
  - epochs run: {len(train_losses)}
  - best validation loss: {best_val_loss:.4f}

test performance:
  - test loss: {val_loss:.4f}

nlg metrics:
  - findings: bleu-1={nlg_metrics['findings']['bleu1']:.4f}, rouge-l={nlg_metrics['findings']['rougeL']:.4f}, bertscore={nlg_metrics['findings']['bertscore']:.4f}
  - impression: bleu-1={nlg_metrics['impression']['bleu1']:.4f}, rouge-l={nlg_metrics['impression']['rougeL']:.4f}, bertscore={nlg_metrics['impression']['bertscore']:.4f}
  - full: bleu-1={nlg_metrics['full']['bleu1']:.4f}, rouge-l={nlg_metrics['full']['rougeL']:.4f}, bertscore={nlg_metrics['full']['bertscore']:.4f}

clinical metrics (chexbert):
  - findings f1_macro: {ce_results['findings']['f1_macro']:.4f}
  - impression f1_macro: {ce_results['impression']['f1_macro']:.4f}
  - full f1_macro: {ce_results['full']['f1_macro']:.4f}

qualitative outputs:
  - dev: {os.path.join(config['train']['log_dir'], 'qualitative_dev.csv')}
  - test: {os.path.join(config['train']['log_dir'], 'qualitative_test.csv')}

output directory: {config['train']['checkpoint_dir']}
====================================================================
"""

print(summary)

with open(os.path.join(config['train']['log_dir'], 'training_summary.txt'), 'w') as f:
    f.write(summary)

print("\nall done. stage-meeting training and evaluation complete.")

##